In [2]:
!uv pip install --quiet tokenizers
!uv pip install --quiet datasets

## Tokenizer

In [3]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
from datasets import load_dataset
from typing import Iterator

In [4]:
class BPETokenizer:
    """
    BPE tokenizer backed by HuggingFace `tokenizers`.
    """

    SPECIAL_TOKENS = [
        "[UNK]", # unknown token, used for out-of-vocabulary words
        "[PAD]", # padding token, used to pad sequences to the same length in a batch
        "[BOS]", # init token, used to indicate the start of a sequence
        "[EOS]"  # stop token, used to indicate the end of a sequence
    ]

    def __init__(self, vocab_size: int = 32768):
        self.vocab_size = vocab_size
        self._tokenizer: Tokenizer | None = None

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------

    def _batch_iterator(self, dataset, text_column: str = "text", batch_size: int = 1000) -> Iterator[list[str]]:
        for i in range(0, len(dataset), batch_size):
            yield dataset[i : i + batch_size][text_column]

    def train(self, dataset, text_column: str = "text") -> None:
        tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
        tokenizer.pre_tokenizer = Whitespace()

        trainer = BpeTrainer(
            vocab_size=self.vocab_size,
            special_tokens=self.SPECIAL_TOKENS,
            min_frequency=2,
            show_progress=True,
        )
        tokenizer.train_from_iterator(
            self._batch_iterator(dataset, text_column),
            trainer=trainer,
            length=len(dataset),
        )
        
        bos_id = tokenizer.token_to_id("[BOS]")
        eos_id = tokenizer.token_to_id("[EOS]")

        tokenizer.post_processor = TemplateProcessing(
            single="[BOS] $A [EOS]",
            special_tokens=[("[BOS]", bos_id), ("[EOS]", eos_id)],
        )

        self._tokenizer = tokenizer

    # ------------------------------------------------------------------
    # Encode / decode
    # ------------------------------------------------------------------

    def encode(self, text: str) -> list[int]:
        self._require_trained()
        return self._tokenizer.encode(text).ids

    def decode(self, ids: list[int]) -> str:
        self._require_trained()
        return self._tokenizer.decode(ids)

    def encode_batch(self, texts: list[str]) -> list[list[int]]:
        self._require_trained()
        return [enc.ids for enc in self._tokenizer.encode_batch(texts)]

    # ------------------------------------------------------------------
    # Persistence
    # ------------------------------------------------------------------

    def save(self, path: str) -> None:
        self._require_trained()
        self._tokenizer.save(path)
        print(f"Tokenizer saved → {path}")

    @classmethod
    def load(cls, path: str) -> "BPETokenizer":
        obj = cls()
        obj._tokenizer = Tokenizer.from_file(path)
        obj.vocab_size = obj._tokenizer.get_vocab_size()
        return obj

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    def _require_trained(self) -> None:
        if self._tokenizer is None:
            raise RuntimeError("Tokenizer has not been trained yet. Call .train() first.")

    @property
    def vocab_size_actual(self) -> int:
        self._require_trained()
        return self._tokenizer.get_vocab_size()

    def __repr__(self) -> str:
        trained = self._tokenizer is not None
        size = self.vocab_size_actual if trained else "?"
        return f"BPETokenizer(vocab_size={size}, trained={trained})"

## Usage example

In [5]:
# Load a small slice of the dataset
dataset = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT", split="train", streaming=False)
small = dataset.select(range(1000000))

# Train
tok = BPETokenizer(vocab_size=32768)
tok.train(small)
print(tok)

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/102 [00:00<?, ?it/s]




BPETokenizer(vocab_size=32768, trained=True)


In [6]:
sample = "The quick brown fox jumps over the lazy dog."
ids = tok.encode(sample)
print("IDs :", ids)
print("Decoded:", tok.decode(ids))

IDs : [2, 10128, 11415, 15229, 20932, 29467, 10318, 10065, 24267, 12027, 17, 3]
Decoded: The quick brown fox jumps over the lazy dog .
